# Construyendo aplicaciones de generación de imágenes (OpenAI)

Los LLMs no solo generan texto. También puedes generar imágenes a partir de descripciones de texto. Las imágenes como modalidad son útiles en MedTech, arquitectura, turismo, desarrollo de juegos, marketing y más. En esta lección trabajamos con los modelos **GPT Image** de hoy en la plataforma OpenAI.

## Objetivos de aprendizaje

Al final de esta lección serás capaz de:

- Explicar qué es la generación de imágenes y dónde es útil.
- Entender la familia de modelos `gpt-image` y cómo difiere de los modelos heredados DALL·E.
- Construir una aplicación de generación de imágenes y editar imágenes.

## ¿Qué es la generación de imágenes?

Los modelos de generación de imágenes crean imágenes a partir de un texto de entrada. Los modelos modernos como `gpt-image` aprenden la relación entre texto e imágenes durante el entrenamiento, y luego transforman iterativamente ruido aleatorio en una imagen que coincide con tu texto.

Dos familias conocidas de modelos de imagen son:

- **`gpt-image` (OpenAI)** — la generación actual usada en esta lección. Soporta generación de imagen desde texto y edición de imágenes (pintura con máscara).
- **Midjourney** — un modelo popular de terceros con su propio servicio y flujo de trabajo basado en Discord.

> Los modelos de imagen antiguos de OpenAI — **DALL·E 2** y **DALL·E 3** — son heredados. DALL·E 3 ya no está disponible para nuevos despliegues, y características como `create_variation` solo existían en DALL·E 2. Usa los modelos `gpt-image` para nuevas aplicaciones.

> **Importante:** los modelos `gpt-image` devuelven la imagen generada como **base64** (`b64_json`), no como una URL. Tu código decodifica la cadena base64 a bytes y la guarda — no hay URL de imagen para descargar.


## Construyendo tu primera aplicación de generación de imágenes

¿Qué se necesita para construir una aplicación de generación de imágenes? Necesitas las siguientes bibliotecas:

- **python-dotenv**, se recomienda encarecidamente usar esta biblioteca para mantener tus secretos en un archivo *.env* separado del código.
- **openai**, esta biblioteca es la que usarás para interactuar con la API de OpenAI.
- **pillow**, para trabajar con imágenes en Python.


1. Crea un archivo *.env* con el siguiente contenido:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Recolecta las bibliotecas anteriores en un archivo llamado *requirements.txt* de la siguiente manera:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. A continuación, crea un entorno virtual e instala las bibliotecas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, use the following commands to create and activate your virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add the following code in file called *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Let's explain this code:

- First, we import the libraries we need, including the OpenAI library, the dotenv library, the base64 module, and the Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- After that, we create the client, which reads the API key from your ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Next, we generate the image:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models return the image as a **base64** string in `data[0].b64_json`. We decode it to bytes and write it to a file — there's no URL to download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lastly, we open the image and use the standard image viewer to display it:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### More details on generating the image

Let's look at the parameters of `images.generate`:

- **model**, is the image model to use (for example `gpt-image-2`).
- **prompt**, is the text prompt used to generate the image. Here it's "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, is the size of the generated image (for example `1024x1024`, `1536x1024`, `1024x1536`, or `"auto"`).
- **n**, is the number of images generated. Here we generate one.

> Image models don't take a `temperature` parameter — that's a text-generation control. To get variety, call the API again; to reduce variety, make your prompt more specific.

## Additional capabilities of image generation

You've seen how to generate an image with a few lines of Python. `gpt-image` models can also **edit** an existing image. By providing an image, an optional **mask** (which marks the area to change), and a prompt, you can alter part of an image — for example, add a hat to our bunny.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

The base image contains just the rabbit; the final image has the hat on the rabbit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
